# 04 Transfer Learning with MobileNetV2

## Goal

Use a model pretrained on ImageNet as a feature extractor for CIFAR-10. Transfer learning can reduce the amount of data and training time needed compared with learning every visual feature from scratch.

MobileNetV2 expects larger images than CIFAR-10's 32 × 32 images. We will resize images to 96 × 96 and apply the preprocessing expected by MobileNetV2.

This notebook may take longer on CPU. If needed, reduce the number of training examples or epochs for the workshop.


In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10

SEED = 42
keras.utils.set_random_seed(SEED)

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.squeeze()

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(x_all))
val_idx, train_idx = idx[:5000], idx[5000:]

# Resize as part of the pipeline to avoid storing a second resized copy of all images.
def prepare(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (96, 96))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_all[train_idx], y_all[train_idx]))
    .shuffle(len(train_idx), seed=SEED)
    .map(prepare, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(64).prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_all[val_idx], y_all[val_idx]))
    .map(prepare, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(64).prefetch(tf.data.AUTOTUNE)
)


2026-11-01 18:03:47.791034: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-01 18:03:47.887473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793556227.924231    4083 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793556227.934070    4083 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793556227.999675    4083 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## 1. Load the pretrained base

We freeze the pretrained feature extractor initially. Only the new classification head will be trained. The ImageNet weights download the first time this cell runs.


In [2]:
base_model = keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=(96, 96, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(10, activation="softmax")(x)

transfer_model = keras.Model(inputs, outputs, name="cifar10_mobilenetv2")

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

transfer_model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 2. Train the classifier head

We use a short run for the workshop. Accuracy depends on compute, training duration, and the pretrained weights.


In [3]:
history_head = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=1, restore_best_weights=True
        )
    ],
)


Epoch 1/3


/home/ashtosh/miniconda3/envs/cubesat-ml/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793556248.628424    4202 service.cc:152] XLA service 0x7d3470002980 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793556248.628559    4202 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6
2026-11-01 18:04:08.829510: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1793556249.749339    4202 cuda_dnn.cc:529] Loaded cuDNN version 90300


 13/704 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.2175 - loss: 2.5250 

I0000 00:00:1793556256.796096    4202 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7832 - loss: 0.6418

2026-11-01 18:04:33.843800: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1156', 4 bytes spill stores, 4 bytes spill loads

2026-11-01 18:04:33.976059: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1156', 60 bytes spill stores, 60 bytes spill loads

2026-11-01 18:04:34.235434: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1156', 948 bytes spill stores, 948 bytes spill loads

2026-11-01 18:04:34.303018: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1156', 1180 bytes spill stores, 1244 bytes spill loads

2026-11-01 18:04:37.461552: I extern

704/704 ━━━━━━━━━━━━━━━━━━━━ 33s 31ms/step - accuracy: 0.7832 - loss: 0.6418 - val_accuracy: 0.8528 - val_loss: 0.4182
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.8442 - loss: 0.4512 - val_accuracy: 0.8596 - val_loss: 0.4043
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 23s 18ms/step - accuracy: 0.8545 - loss: 0.4241 - val_accuracy: 0.8636 - val_loss: 0.3883


## 3. Optional: fine-tune a small part of the base

Fine-tuning can improve performance, but it can also overfit or destabilize pretrained features. Unfreeze only a small number of top layers and use a much lower learning rate. Skip this section if time or compute is limited.


In [4]:
# Optional fine-tuning. Run only if the first training stage completed.
base_model.trainable = True

# Keep most layers frozen; unfreeze only the final 20 layers.
for layer in base_model.layers[:-20]:
    layer.trainable = False

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_finetune = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=1, restore_best_weights=True
        )
    ],
)


Epoch 1/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 32s 32ms/step - accuracy: 0.8094 - loss: 0.5864 - val_accuracy: 0.8552 - val_loss: 0.4405
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.8572 - loss: 0.4291 - val_accuracy: 0.8688 - val_loss: 0.3855


In [5]:
os.makedirs("../artifacts", exist_ok=True)
transfer_model.save("../artifacts/cifar10_mobilenetv2.keras")
print("Saved: ../artifacts/cifar10_mobilenetv2.keras")


Saved: ../artifacts/cifar10_mobilenetv2.keras


## Reflection

- What does the pretrained model already know how to extract?
- Why freeze the base first?
- Why use a smaller learning rate for fine-tuning?
- Why must inference use the same resizing and preprocessing as training?

**Next:** evaluate the models using metrics and inspect their errors.
